# Giáo trình Dữ liệu lớn – Chương 8

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài OpenJDK 17, PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume `/Volumes/workspace/default/du_lieu/` do người học tự tạo và thay `data/` bằng đường dẫn này; tính toán serverless của Free Edition không hỗ trợ API RDD/`SparkContext` và `cache()`/`persist()` (xem [README](https://github.com/mocminh/bigdata_code#databricks-free-edition)).

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch08/doan_ma_*.py`.


> **Lưu ý:** Chương 8 là nội dung mở rộng; các đoạn mã cần GPU, TensorFlow 2.15/PyTorch 2.4/Horovod/MLflow (`requirements-ch08.txt`) và mô hình/dữ liệu của người học. Notebook này chỉ để đọc và chỉnh sửa, không chạy trực tiếp trên Colab miễn phí.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
# Buoc 1: cai OpenJDK 17 (Spark 3.5 ho tro Java 8/11/17)
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# Buoc 2: cai dat PySpark tu PyPI (ghim phien ban theo Bang 2.3)
!pip install -q pyspark==3.5.7
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch08").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 8.1. Suy luận phân tán mô hình Keras trên Spark bằng mapInPandas.


> **Khung / minh họa – không chạy trực tiếp:** cần TensorFlow + mô hình đã huấn luyện (GPU).


In [ ]:
import numpy as np
import pandas as pd

MODEL_PATH = "models/image_classifier.keras"

def predict_batch(iterator):
    # Nap mo hinh MOT LAN cho moi partition, dung lai cho moi lo
    import tensorflow as tf
    model = tf.keras.models.load_model(MODEL_PATH)
    for pdf in iterator:
        X = np.stack(pdf["features"].to_numpy())
        probs = model.predict(X, batch_size=256, verbose=0)
        yield pd.DataFrame({
            "id": pdf["id"],
            "label_pred": probs.argmax(axis=1),
            "confidence": probs.max(axis=1)})

df_pred = df_features.mapInPandas(
    predict_batch,
    schema="id long, label_pred int, confidence float")
(df_pred.write.mode("overwrite")
    .parquet("output/predictions"))

## Đoạn mã 8.2. Khung huấn luyện phân tán PyTorch với TorchDistributor (Spark 3.4 trở lên).


> **Khung / minh họa – không chạy trực tiếp:** cần cụm GPU và hàm build_model, make_dataloader của người học.


In [ ]:
from pyspark.ml.torch.distributor import TorchDistributor

def train_fn(lr, epochs):
    import os
    import torch
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP

    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    local_rank = int(os.environ["LOCAL_RANK"])  # GPU cuc bo
    device = torch.device(f"cuda:{local_rank}")

    model = build_model().to(device)  # do nguoi dung dinh nghia
    model = DDP(model, device_ids=[local_rank])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # moi rank doc mot shard du lieu rieng
        for X, y in make_dataloader(rank):
            optimizer.zero_grad()
            loss = torch.nn.functional.cross_entropy(
                model(X.to(device)), y.to(device))
            loss.backward()        # DDP tu dong all-reduce gradient
            optimizer.step()
    if rank == 0:
        torch.save(model.module.state_dict(), "models/net.pt")

distributor = TorchDistributor(num_processes=8,
                               local_mode=False, use_gpu=True)
distributor.run(train_fn, lr=1e-3, epochs=10)

## Đoạn mã 8.3. Khung huấn luyện Keras phân tán với Horovod.


> **Khung / minh họa – không chạy trực tiếp:** cần Horovod + GPU. Dự án Horovod đã lưu trữ (archived) từ 9/2026, 0.28.1 là phiên bản cuối; tài liệu chính thức chỉ ghi nhận tương thích tới TensorFlow 2.12, nên cần tự kiểm thử trước khi dùng với môi trường đã ghim (Mục 8.3.2).


In [ ]:
import tensorflow as tf
import horovod.tensorflow.keras as hvd

hvd.init()                          # 1. Khoi tao Horovod

# 2. Gan (pin) moi tien trinh voi mot GPU cuc bo
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.set_visible_devices(gpus[hvd.local_rank()], "GPU")

model = build_model()
# 3. Scale learning rate theo so worker
opt = tf.keras.optimizers.SGD(learning_rate=0.01 * hvd.size())
# 4. Boc optimizer de dong bo gradient bang All-Reduce
opt = hvd.DistributedOptimizer(opt)
model.compile(optimizer=opt,
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

callbacks = [
    # 5. Phat trong so ban dau tu rank 0 toi moi worker
    hvd.callbacks.BroadcastGlobalVariablesCallback(0),
    hvd.callbacks.MetricAverageCallback(),
]
if hvd.rank() == 0:                 # chi rank 0 ghi checkpoint
    callbacks.append(
        tf.keras.callbacks.ModelCheckpoint("models/ckpt/best.keras"))

model.fit(train_dataset, epochs=20, callbacks=callbacks,
          verbose=1 if hvd.rank() == 0 else 0)

## Đoạn mã 8.4. Theo dõi thí nghiệm và đăng ký mô hình Keras với MLflow.


> **Khung / minh họa – không chạy trực tiếp:** cần TensorFlow + MLflow server. Mô hình được nạp theo bí danh (alias) `@champion` của Model Registry; các giai đoạn Staging/Production bị đánh dấu lỗi thời từ MLflow 2.9 và Unity Catalog trên Databricks không hỗ trợ. Tên thí nghiệm `/exp/image-classifier` trong sách được đổi thành `image-classifier` để chạy với MLflow cục bộ.


In [ ]:
import mlflow
import mlflow.tensorflow

mlflow.set_experiment("image-classifier")
mlflow.tensorflow.autolog()   # tu dong log tham so, metric, mo hinh

with mlflow.start_run(run_name="cnn_baseline"):
    mlflow.log_param("batch_size", 128)
    mlflow.log_param("learning_rate", 1e-3)
    history = model.fit(train_ds, validation_data=val_ds, epochs=20)
    best_acc = max(history.history["val_accuracy"])
    mlflow.log_metric("best_val_accuracy", best_acc)
    mlflow.log_artifact("confusion_matrix.png")
    mlflow.tensorflow.log_model(
        model, artifact_path="model",
        registered_model_name="cnn_classifier")

# Gan bi danh champion cho phien ban da kiem dinh (vi du ban 1)
client = mlflow.MlflowClient()
client.set_registered_model_alias("cnn_classifier", "champion", 1)
# Nap mo hinh theo bi danh, khong can biet so phien ban
model_prod = mlflow.pyfunc.load_model(
    "models:/cnn_classifier@champion")
preds = model_prod.predict(new_data)

## Đoạn mã 8.5. Suy luận phân tán bằng mapInPandas.


> **Khung / minh họa – không chạy trực tiếp:** cần TensorFlow + mô hình đã huấn luyện.


In [ ]:
import numpy as np
import pandas as pd

def du_doan(cac_lo):
    import tensorflow as tf
    # nap mo hinh MOT LAN cho moi partition
    model = tf.keras.models.load_model("models/model.h5")
    for lo in cac_lo:              # moi lo la mot pandas DataFrame
        X = np.stack(lo["dac_trung"].to_numpy())
        lo["nhan"] = model.predict(X).argmax(axis=1)
        yield lo[["id", "nhan"]]

ketqua = df.mapInPandas(du_doan, schema="id string, nhan int")
ketqua.write.mode("overwrite").parquet("output/ketqua_du_doan")